In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.impute import KNNImputer
from sklearn.metrics import roc_auc_score, confusion_matrix
from sklearn.model_selection import train_test_split
from joblib import Parallel, delayed

# ======================
# Data Loading
# ======================
diagnoses = pd.read_csv('diagnoses.csv')
movement_features = pd.read_csv('movement_features.csv')
validation = pd.read_csv('validation.csv')

# ======================
# Preprocessing
# ======================
validation = validation[['ID', 'Country']]

country_map = {'England': 1, 'Scotland': 2, 'Wales': 3}
validation['split'] = validation['Country'].map(country_map)

# Merge split info
movement_features = pd.merge(validation[['ID', 'split']],
                             movement_features,
                             on='ID',
                             how='right')

diagnoses = pd.merge(validation[['ID', 'split']],
                     diagnoses,
                     on='ID',
                     how='right')

# Sort
movement_features = movement_features.sort_values('ID').reset_index(drop=True)
diagnoses = diagnoses.sort_values('ID').reset_index(drop=True)

# Remove all-NaN rows
valid_ids = movement_features.loc[
    ~movement_features.iloc[:, 2:].isna().all(axis=1),
    'ID']

movement_features = movement_features[movement_features['ID'].isin(valid_ids)]
diagnoses = diagnoses[diagnoses['ID'].isin(valid_ids)]

# ======================
# Imputation
# ======================
imputer = KNNImputer(n_neighbors=5)
movement_features.iloc[:, 2:] = imputer.fit_transform(movement_features.iloc[:, 2:])

# ======================
# Train/Test Split
# ======================
train_mask = movement_features['split'] == 1
test_mask = movement_features['split'].isin([2, 3])

train_X = movement_features.iloc[:, 2:][train_mask].values.reshape(-1, 48, 4)
test_X = movement_features.iloc[:, 2:][test_mask].values.reshape(-1, 48, 4)

train_y_df = diagnoses[train_mask]
test_y_df = diagnoses[test_mask]

# ======================
# Model Definition
# ======================
class TMRM(nn.Module):
    def __init__(self, input_size=4, hidden_size=32):
        super(TMRM, self).__init__()

        self.lstm_workday = nn.ModuleList([
            nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
            for _ in range(input_size)
        ])

        self.lstm_restday = nn.ModuleList([
            nn.LSTM(input_size=1, hidden_size=hidden_size, batch_first=True)
            for _ in range(input_size)
        ])

        self.attn_workday = nn.ModuleList([
            nn.MultiheadAttention(embed_dim=hidden_size, num_heads=2)
            for _ in range(input_size)
        ])

        self.attn_restday = nn.ModuleList([
            nn.MultiheadAttention(embed_dim=hidden_size, num_heads=2)
            for _ in range(input_size)
        ])

        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_size * input_size * 2, 1)

    def attention_block(self, x, attn_layer):
        x = x.permute(1, 0, 2)
        attn_out, _ = attn_layer(x, x, x)
        attn_out = attn_out.permute(1, 0, 2)
        return torch.mean(attn_out, dim=1)

    def forward(self, workday, restday):
        workday_feats = []
        for i in range(workday.size(2)):
            out, _ = self.lstm_workday[i](workday[:, :, i].unsqueeze(2))
            out = self.attention_block(out, self.attn_workday[i])
            workday_feats.append(out)

        restday_feats = []
        for i in range(restday.size(2)):
            out, _ = self.lstm_restday[i](restday[:, :, i].unsqueeze(2))
            out = self.attention_block(out, self.attn_restday[i])
            restday_feats.append(out)

        features = torch.cat(workday_feats + restday_feats, dim=1)
        features = self.dropout(features)

        return self.fc(features)

# ======================
# Training Function
# ======================
def train_model(disease):

    y = train_y_df[disease]
    valid_idx = y.isin([0, 1])
    y = y[valid_idx]

    X = train_X[valid_idx]

    workday = X[:, :24, :]
    restday = X[:, 24:, :]

    workday = torch.tensor(workday, dtype=torch.float32)
    restday = torch.tensor(restday, dtype=torch.float32)
    y = torch.tensor(y.values, dtype=torch.float32).unsqueeze(1)

    wd_tr, wd_val, rd_tr, rd_val, y_tr, y_val = train_test_split(
        workday, restday, y,
        test_size=0.2,
        random_state=42,
        stratify=y.numpy()
    )

    train_loader = DataLoader(
        TensorDataset(wd_tr, rd_tr, y_tr),
        batch_size=128,
        shuffle=True
    )

    model = TMRM()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.BCEWithLogitsLoss()

    # Training
    model.train()
    for epoch in range(20):
        total_loss = 0
        for wd, rd, labels in train_loader:
            optimizer.zero_grad()
            out = model(wd, rd)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        print(f"{disease} Epoch {epoch+1}: {total_loss/len(train_loader):.4f}")

    # Evaluation
    model.eval()
    with torch.no_grad():
        pred = model(wd_val, rd_val).squeeze()
        prob = torch.sigmoid(pred)
        pred_label = (prob > 0.5).float()

        auc = roc_auc_score(y_val.numpy(), prob.numpy())

        cm = confusion_matrix(y_val.numpy(), pred_label.numpy())
        TP = cm[1, 1]
        FN = cm[1, 0]
        sensitivity = TP / (TP + FN) if (TP + FN) else 0

    return disease, auc, sensitivity, model

# ======================
# Parallel Training
# ======================
def run_training(disease_cols, n_jobs=8):

    results = Parallel(n_jobs=n_jobs)(
        delayed(train_model)(d) for d in disease_cols
    )

    for d, auc, sen, _ in results:
        print(f"{d}: AUC={auc:.4f}, Sensitivity={sen:.4f}")

    return results

# ======================
# Main
# ======================
if __name__ == "__main__":

    disease_columns = train_y_df.columns[2:]

    run_training(disease_columns)